# Dataset Download + Metadata Builder

This notebook downloads the two required zips, extracts the APPA-REAL images,
and rebuilds metadata CSVs in the format used by this repo.

It produces:
- `datasets/appa-real-dataset_v2/{train_data,valid_data,test_data}` (images only)
- `datasets/appa-real-dataset_v2/labels_metadata_{train,valid,test}.csv`

Run from the repo root.


## Config
Adjust paths or toggles as needed before running the rest.


In [1]:
from pathlib import Path
import shutil
import urllib.request
import zipfile

import pandas as pd

REPO_ROOT = Path('.').resolve()
DATASET_ROOT = REPO_ROOT / 'datasets' / 'appa-real-dataset_v2'
DOWNLOAD_ROOT = REPO_ROOT / 'datasets' / '_downloads'

APPA_REAL_URL = 'https://data.chalearnlap.cvc.uab.cat/AppaRealAge/appa-real-release.zip'
ALLCATEGORIES_URL = 'http://sergioescalera.com/wp-content/uploads/2018/06/allcategories_trainvalidtest_split.zip'

USE_EXISTING_DOWNLOADS = False
EXISTING_DOWNLOADS_DIR = Path.home() / 'Downloads'

DROP_MISSING_AGE = True
AGE_COLUMN = 'real_age'

CLEAN_TARGET_DIRS = False  # Set True to delete existing train/valid/test_data before copy.
CLEANUP_DOWNLOADS = True  # Set True to remove downloaded zips and extracted folders.


## Helpers


In [2]:
def _format_eta(seconds: float | None) -> str:
    if seconds is None:
        return '?:??'
    seconds = max(int(seconds), 0)
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60
    if hours:
        return f'{hours}:{minutes:02d}:{secs:02d}'
    return f'{minutes:02d}:{secs:02d}'


def _download(url: str, dest: Path, chunk_size: int = 8 * 1024 * 1024) -> Path:
    import time

    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        print(f'Already downloaded: {dest}')
        return dest
    print(f'Downloading {url} -> {dest}')
    downloaded = 0
    start = time.monotonic()
    with urllib.request.urlopen(url) as response, open(dest, 'wb') as f:
        total = response.getheader('Content-Length')
        total = int(total) if total else None
        while True:
            chunk = response.read(chunk_size)
            if not chunk:
                break
            f.write(chunk)
            downloaded += len(chunk)

            elapsed = max(time.monotonic() - start, 1e-6)
            speed = downloaded / elapsed
            speed_mb = speed / (1024 * 1024)
            downloaded_mb = downloaded / (1024 * 1024)

            if total:
                total_mb = total / (1024 * 1024)
                pct = downloaded / total
                width = 30
                filled = int(pct * width)
                bar = '#' * filled + '-' * (width - filled)
                eta = (total - downloaded) / speed if speed > 0 else None
                eta_str = _format_eta(eta)
                print(
                    f'\r[{bar}] {downloaded_mb:.1f}/{total_mb:.1f} MB {pct*100:5.1f}% '
                    f'{speed_mb:.1f} MB/s ETA {eta_str}',
                    end='',
                    flush=True,
                )
            else:
                print(
                    f'\rDownloaded {downloaded_mb:.1f} MB {speed_mb:.1f} MB/s',
                    end='',
                    flush=True,
                )
    print()
    return dest
def _extract(zip_path: Path, dest_dir: Path) -> Path:
    if dest_dir.exists() and any(dest_dir.iterdir()):
        print(f'Already extracted: {dest_dir}')
        return dest_dir
    dest_dir.mkdir(parents=True, exist_ok=True)
    print(f'Extracting {zip_path} -> {dest_dir}')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(dest_dir)
    return dest_dir


def _resolve_root(dest_dir: Path) -> Path:
    candidates = [p for p in dest_dir.iterdir() if p.is_dir() and p.name != '__MACOSX']
    if len(candidates) == 1 and not any(p.is_file() for p in dest_dir.iterdir()):
        return candidates[0]
    return dest_dir


def _locate_file(root: Path, filename: str) -> Path:
    direct = root / filename
    if direct.exists():
        return direct
    matches = sorted(root.rglob(filename), key=lambda p: len(p.parts))
    if matches:
        return matches[0]
    raise FileNotFoundError(f'Missing {filename} under {root}')


def _locate_dir(root: Path, dirname: str) -> Path:
    direct = root / dirname
    if direct.is_dir():
        return direct
    matches = sorted([p for p in root.rglob(dirname) if p.is_dir()], key=lambda p: len(p.parts))
    if matches:
        return matches[0]
    raise FileNotFoundError(f'Missing directory {dirname} under {root}')


def _normalize_image_id(value) -> str:
    s = str(value).strip()
    if s.lower().endswith('.jpg'):
        s = s[:-4]
    if s.isdigit() and len(s) < 6:
        s = s.zfill(6)
    return s


def _normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    return df


def _select_column(columns, preferences):
    for name in preferences:
        if name in columns:
            return name
    return None


## Download and extract
If `USE_EXISTING_DOWNLOADS = True`, the notebook looks for zip files in `EXISTING_DOWNLOADS_DIR`.


In [3]:
if USE_EXISTING_DOWNLOADS:
    appa_zip = EXISTING_DOWNLOADS_DIR / 'appa-real-release.zip'
    meta_zip = EXISTING_DOWNLOADS_DIR / 'allcategories_trainvalidtest_split.zip'
else:
    appa_zip = _download(APPA_REAL_URL, DOWNLOAD_ROOT / 'appa-real-release.zip')
    meta_zip = _download(ALLCATEGORIES_URL, DOWNLOAD_ROOT / 'allcategories_trainvalidtest_split.zip')

appa_dir = _extract(appa_zip, DOWNLOAD_ROOT / 'appa-real-release')
meta_dir = _extract(meta_zip, DOWNLOAD_ROOT / 'allcategories_trainvalidtest_split')

appa_root = _resolve_root(appa_dir)
meta_root = _resolve_root(meta_dir)

print('APPA-REAL root:', appa_root)
print('All-categories root:', meta_root)


Already downloaded: /home/fox/Desktop/Projects/Amrita RLHF/Diffusion_RL_Abi/datasets/_downloads/appa-real-release.zip
Already downloaded: /home/fox/Desktop/Projects/Amrita RLHF/Diffusion_RL_Abi/datasets/_downloads/allcategories_trainvalidtest_split.zip
Already extracted: /home/fox/Desktop/Projects/Amrita RLHF/Diffusion_RL_Abi/datasets/_downloads/appa-real-release
Already extracted: /home/fox/Desktop/Projects/Amrita RLHF/Diffusion_RL_Abi/datasets/_downloads/allcategories_trainvalidtest_split
APPA-REAL root: /home/fox/Desktop/Projects/Amrita RLHF/Diffusion_RL_Abi/datasets/_downloads/appa-real-release/appa-real-release
All-categories root: /home/fox/Desktop/Projects/Amrita RLHF/Diffusion_RL_Abi/datasets/_downloads/allcategories_trainvalidtest_split


## Copy images into `datasets/appa-real-dataset_v2`
Only `.jpg` images are copied (excluding `*_face.jpg`).


In [4]:
split_map = {
    'train': 'train_data',
    'valid': 'valid_data',
    'test': 'test_data',
}

if CLEAN_TARGET_DIRS:
    for dst_name in split_map.values():
        target = DATASET_ROOT / dst_name
        if target.exists():
            print(f'Removing {target}')
            shutil.rmtree(target)

for split, dst_name in split_map.items():
    src_dir = _locate_dir(appa_root, split)
    dst_dir = DATASET_ROOT / dst_name
    dst_dir.mkdir(parents=True, exist_ok=True)

    count = 0
    for img_path in src_dir.rglob('*.jpg'):
        if img_path.name.endswith('_face.jpg'):
            continue
        shutil.copy2(img_path, dst_dir / img_path.name)
        count += 1

    print(f'Copied {count} images -> {dst_dir}')


Copied 4113 images -> /home/fox/Desktop/Projects/Amrita RLHF/Diffusion_RL_Abi/datasets/appa-real-dataset_v2/train_data
Copied 1500 images -> /home/fox/Desktop/Projects/Amrita RLHF/Diffusion_RL_Abi/datasets/appa-real-dataset_v2/valid_data
Copied 1978 images -> /home/fox/Desktop/Projects/Amrita RLHF/Diffusion_RL_Abi/datasets/appa-real-dataset_v2/test_data


## Build `labels_metadata_{split}.csv`
We take gender/ethnicity/emotion from the all-categories CSVs,
and age from the APPA-REAL `gt_{split}.csv` real-age column.


In [5]:
def load_age_map(split: str) -> pd.DataFrame:
    path = _locate_file(appa_root, f'gt_{split}.csv')

    df = pd.read_csv(path)
    df = _normalize_columns(df)
    if 'file_name' in df.columns and 'filename' not in df.columns:
        df = df.rename(columns={'file_name': 'filename'})

    id_col = _select_column(df.columns, [
        'image',
        'image_id',
        'imageid',
        'file',
        'filename',
        'image_name',
        'file_name',
    ])

    if id_col is None or AGE_COLUMN not in df.columns:
        raise ValueError(
            f'Unable to infer id/age columns in {path}. '
            f'Columns: {sorted(df.columns)}'
        )

    df = df[[id_col, AGE_COLUMN]].copy()
    df['imageId'] = df[id_col].map(_normalize_image_id)
    df['age'] = df[AGE_COLUMN]

    # gt_*.csv can include multiple ratings per image; average if needed.
    if df['imageId'].duplicated().any():
        df = df.groupby('imageId', as_index=False)['age'].mean()
    else:
        df = df[['imageId', 'age']]

    print(f"Using age column '{AGE_COLUMN}' from {path.name}")
    return df


In [6]:
def load_allcategories(split: str) -> pd.DataFrame:
    path = _locate_file(meta_root, f'allcategories_{split}.csv')
    df = pd.read_csv(path)
    df = _normalize_columns(df)

    required = {'file', 'gender', 'race', 'happiness'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'Missing columns in {path}: {sorted(missing)}')

    df = df.rename(columns={
        'file': 'filename',
        'race': 'ethnicity',
        'happiness': 'emotion',
    })
    df['imageId'] = df['filename'].map(_normalize_image_id)
    df = df[['imageId', 'gender', 'ethnicity', 'emotion']]
    return df


def load_age_map(split: str) -> pd.DataFrame:
    path = _locate_file(appa_root, f'gt_{split}.csv')

    df = pd.read_csv(path)
    df = _normalize_columns(df)
    if 'file_name' in df.columns and 'filename' not in df.columns:
        df = df.rename(columns={'file_name': 'filename'})

    id_col = _select_column(df.columns, [
        'image',
        'image_id',
        'imageid',
        'file',
        'filename',
        'image_name',
        'file_name',
    ])

    if id_col is None or AGE_COLUMN not in df.columns:
        raise ValueError(
            f'Unable to infer id/age columns in {path}. '
            f'Columns: {sorted(df.columns)}'
        )

    df = df[[id_col, AGE_COLUMN]].copy()
    df['imageId'] = df[id_col].map(_normalize_image_id)
    df['age'] = df[AGE_COLUMN]

    if df['imageId'].duplicated().any():
        df = df.groupby('imageId', as_index=False)['age'].mean()
    else:
        df = df[['imageId', 'age']]

    print(f"Using age column '{AGE_COLUMN}' from {path.name}")
    return df



def build_labels(split: str) -> pd.DataFrame:
    meta = load_allcategories(split)
    ages = load_age_map(split)

    merged = meta.merge(ages, on='imageId', how='left', validate='1:1')
    missing_age = merged['age'].isna().sum()
    if missing_age:
        print(f'Warning: {missing_age} rows missing age for split={split}')
        if DROP_MISSING_AGE:
            merged = merged.dropna(subset=['age'])

    merged = merged[['imageId', 'age', 'gender', 'ethnicity', 'emotion']]
    return merged


In [7]:
for split in ['train', 'valid', 'test']:
    df = build_labels(split)
    out_path = DATASET_ROOT / f'labels_metadata_{split}.csv'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)
    print(f'Wrote {len(df)} rows -> {out_path}')

    display(df.head())


Using age column 'real_age' from gt_train.csv
Wrote 4113 rows -> /home/fox/Desktop/Projects/Amrita RLHF/Diffusion_RL_Abi/datasets/appa-real-dataset_v2/labels_metadata_train.csv


,imageId,age,gender,ethnicity,emotion
0,000000,4.0,male,caucasian,neutral
1,000001,18.0,female,caucasian,neutral
2,000002,80.0,female,caucasian,slightlyhappy
3,000003,50.0,female,caucasian,happy
4,000004,17.0,female,caucasian,slightlyhappy


Using age column 'real_age' from gt_valid.csv
Wrote 1500 rows -> /home/fox/Desktop/Projects/Amrita RLHF/Diffusion_RL_Abi/datasets/appa-real-dataset_v2/labels_metadata_valid.csv


,imageId,age,gender,ethnicity,emotion
0,004113,29.0,male,afroamerican,happy
1,004114,25.0,male,caucasian,other
2,004115,37.0,male,caucasian,slightlyhappy
3,004116,80.0,female,caucasian,happy
4,004117,25.0,female,caucasian,happy


Using age column 'real_age' from gt_test.csv
Wrote 1978 rows -> /home/fox/Desktop/Projects/Amrita RLHF/Diffusion_RL_Abi/datasets/appa-real-dataset_v2/labels_metadata_test.csv


,imageId,age,gender,ethnicity,emotion
0,005613,19.0,female,caucasian,neutral
1,005614,76.0,male,asian,slightlyhappy
2,005615,40.0,male,asian,happy
3,005616,21.0,male,caucasian,neutral
4,005617,34.0,female,caucasian,neutral


## Validate image/metadata alignment
Checks for missing metadata rows for images and vice versa.


In [8]:
def _list_image_ids(folder: Path) -> set[str]:
    ids = set()
    for ext in ('*.jpg', '*.png', '*.jpeg', '*.JPG', '*.PNG', '*.JPEG'):
        for path in folder.rglob(ext):
            ids.add(_normalize_image_id(path.stem))
    return ids


for split, dst_name in split_map.items():
    images_dir = DATASET_ROOT / dst_name
    labels_path = DATASET_ROOT / f'labels_metadata_{split}.csv'

    if not images_dir.exists():
        print(f'[WARN] Missing images folder: {images_dir}')
        continue
    if not labels_path.exists():
        print(f'[WARN] Missing metadata file: {labels_path}')
        continue

    labels_df = pd.read_csv(labels_path)
    labels_df = _normalize_columns(labels_df)

    if 'imageid' not in labels_df.columns:
        print(f'[WARN] Missing imageId column in {labels_path}')
        continue

    label_ids = { _normalize_image_id(v) for v in labels_df['imageid'] }
    image_ids = _list_image_ids(images_dir)

    missing_meta = sorted(image_ids - label_ids)
    missing_imgs = sorted(label_ids - image_ids)

    print(f'[{split}] images={len(image_ids)} labels={len(label_ids)}')
    print(f'[{split}] missing metadata for images: {len(missing_meta)}')
    print(f'[{split}] metadata without image: {len(missing_imgs)}')

    if missing_meta:
        print(f'[{split}] sample missing metadata IDs: {missing_meta[:10]}')
    if missing_imgs:
        print(f'[{split}] sample metadata-only IDs: {missing_imgs[:10]}')



[train] images=4113 labels=4113
[train] missing metadata for images: 0
[train] metadata without image: 0
[valid] images=2982 labels=1500
[valid] missing metadata for images: 1482
[valid] metadata without image: 0
[valid] sample missing metadata IDs: ['000001', '000002', '000003', '000004', '000005', '000006', '000007', '000008', '000009', '000010']
[test] images=3956 labels=1978
[test] missing metadata for images: 1978
[test] metadata without image: 0
[test] sample missing metadata IDs: ['000001', '000002', '000003', '000004', '000005', '000006', '000007', '000008', '000009', '000010']


## Optional cleanup
Enable `CLEANUP_DOWNLOADS = True` to delete the downloaded zips and extracted folders.


In [9]:
if CLEANUP_DOWNLOADS:
    for path in [appa_zip, meta_zip, appa_dir, meta_dir]:
        if path.exists():
            if path.is_dir():
                shutil.rmtree(path)
            else:
                path.unlink()
            print(f'Removed {path}')

Removed /home/fox/Desktop/Projects/Amrita RLHF/Diffusion_RL_Abi/datasets/_downloads/appa-real-release.zip
Removed /home/fox/Desktop/Projects/Amrita RLHF/Diffusion_RL_Abi/datasets/_downloads/allcategories_trainvalidtest_split.zip
Removed /home/fox/Desktop/Projects/Amrita RLHF/Diffusion_RL_Abi/datasets/_downloads/appa-real-release
Removed /home/fox/Desktop/Projects/Amrita RLHF/Diffusion_RL_Abi/datasets/_downloads/allcategories_trainvalidtest_split
